# UniGuide AI — Week 3 data pipeline (version 1.3)

**Dataset change (v1.1, kept in v1.3):** core set = S01 academic calendar, S02 academic regulations, S06 UG work schedule, S07 matric card FAQ, S08 UTMID FAQ, plus S05 senior-student registration manual. Retired: S03 (2010 advising handbook, quarantined) and S04 (new-student manual, dropped). IDs S01, S02, S05 keep their meaning so earlier chunk IDs stay valid.

**Version 1.2:** the chunk splitter now cuts only after a sentence end (fixes mid-sentence chunk starts in S02).

**Version 1.3:** adds S09, the Semester II 2026/2027 UG work schedule (companion to S06, Semester I).

**OCR policy:** pypdf first. OCR only for the three image regions that carry answer content (S05 p9 slip legend, S08 Q6 and Q7 infographics). OCR output is checked against the page image and kept as `ocr_raw_text` beside the verified text. Pages with personal data (S05 p3, p8 screenshots) are never OCR'd.

In [4]:
# Cell 1 — Workspace setup (RUN FIRST)
import os
from pathlib import Path

BASE_DIR = Path(os.environ.get("UNIGUIDE_BASE", "/content/week3_project"))
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


In [5]:
# Cell 3 — Verify uploaded inputs (self-contained)
EXPECTED_FILES = [
    "utm_academic_calendar_2026_2027_1.pdf",                                              # S01
    "utm_academic_regulations.pdf",                                                       # S02
    "utm_User-Manual_Course-Registration-Senior-Student_update202620271.pdf",             # S05
    "utm_ug_work_schedule_2026_2027.pdf",                                                 # S06
    "utm_ug_work_schedule_sem2_2026_2027.pdf",                                            # S09
    "utm_matric_card_faq_2026.pdf",                                                       # S07
    "utm_utmid_faq_2026.pdf",                                                             # S08
]
files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
missing = [n for n in EXPECTED_FILES if not (RAW / n).is_file()]
assert not missing, f"Missing expected inputs: {missing}"
print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 7
utm_User-Manual_Course-Registration-Senior-Student_update202620271.pdf bytes: 2055792
utm_academic_calendar_2026_2027_1.pdf bytes: 341794
utm_academic_regulations.pdf bytes: 779259
utm_matric_card_faq_2026.pdf bytes: 185309
utm_ug_work_schedule_2026_2027.pdf bytes: 563009
utm_ug_work_schedule_sem2_2026_2027.pdf bytes: 250218
utm_utmid_faq_2026.pdf bytes: 335228
Upload verified. Select your relevant input(s) for Step A3.


In [6]:
# Cell 4 — Inspect source documents (A3)
!pip -q install "pypdf>=5.9,<7"   # tested with 5.9.0 and 6.17.0; pypdf 3.x extracts S06 and S08 differently
import re, csv
from pypdf import PdfReader
import logging; logging.getLogger("pypdf").setLevel(logging.ERROR)

summary = []
for f in sorted(RAW.rglob("*.pdf")):
    reader = PdfReader(str(f))
    texts = [(p.extract_text() or "") for p in reader.pages]
    chars = sum(len(t) for t in texts)
    low = [i + 1 for i, t in enumerate(texts) if len(t.strip()) < 20]
    gid = sum(len(re.findall(r"/gid\d+", t)) for t in texts)
    meta = reader.metadata or {}
    img_pages = []
    for i, p in enumerate(reader.pages, 1):          # pages that carry embedded images (OCR candidates)
        try:
            if len(p.images) > 0: img_pages.append(i)
        except Exception:
            pass

    print("=" * 70)
    print("FILE:", f.name, "| bytes:", f.stat().st_size)
    print("Pages:", len(texts), "| characters:", chars)
    print("Pages with little/no text:", low)
    print("Pages with embedded images:", img_pages)
    print("Garbled glyph codes (/gid...):", gid)
    print("Created:", meta.get("/CreationDate"), "| Modified:", meta.get("/ModDate"))
    sample = next((t for t in texts if t.strip()), "")
    print("--- Sample (first 300 chars of first non-empty page) ---")
    print(sample[:300])

    summary.append({"file": f.name, "bytes": f.stat().st_size, "pages": len(texts), "chars": chars,
                    "low_text_pages": low, "image_pages": img_pages, "gid_tokens": gid})

out = PROCESSED_DIR / "inspection_summary.csv"
with open(out, "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=summary[0].keys())
    w.writeheader(); w.writerows(summary)
print("\nSaved:", out)

FILE: utm_User-Manual_Course-Registration-Senior-Student_update202620271.pdf | bytes: 2055792
Pages: 10 | characters: 2048
Pages with little/no text: []
Pages with embedded images: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Garbled glyph codes (/gid...): 0
Created: D:20261006095121+08'00' | Modified: D:20261006113042+08'00'
--- Sample (first 300 chars of first non-empty page) ---
Step-by-step
COURSE 
REGISTRATION 
Bachelor Degree (Fulltime) 
Senior Student

FILE: utm_academic_calendar_2026_2027_1.pdf | bytes: 341794
Pages: 1 | characters: 3568
Pages with little/no text: []
Pages with embedded images: [1]
Garbled glyph codes (/gid...): 0
Created: D:20261120102928+00'00' | Modified: None
--- Sample (first 300 chars of first non-empty page) ---
DATE PROGRAMME PUBLIC HOLIDAY 
International Undergraduate and 
Postgraduate Students: 
21 - 24 September 2026 (UTM JB) 
26 & 27 September 2026 (UTM KL)
Local Undergraduate Students:      
4 October 2026 (UTM JB & UTM KL)  
Local Postgraduate Students: 
14 -1

In [7]:
# Cell 4b — Readable evidence pages for the inventory
def show(filename_part, pages, n=500):
    f = next(f for f in RAW.rglob("*.pdf") if filename_part in f.name)
    r = PdfReader(str(f))
    for p in pages:
        t = (r.pages[p - 1].extract_text() or "")
        print(f"--- {f.name} | page {p} ---")
        print(t[:n], "\n")

show("academic_calendar", [1])
show("academic_regulations", [40])
show("work_schedule", [1])
show("matric_card_faq", [1], 400)
show("utmid_faq", [1], 400)

--- utm_academic_calendar_2026_2027_1.pdf | page 1 ---
DATE PROGRAMME PUBLIC HOLIDAY 
International Undergraduate and 
Postgraduate Students: 
21 - 24 September 2026 (UTM JB) 
26 & 27 September 2026 (UTM KL)
Local Undergraduate Students:      
4 October 2026 (UTM JB & UTM KL)  
Local Postgraduate Students: 
14 -18 September 2026 (UTM JB) 
26 & 27 September 2026 (UTM KL)
Registration of New Students 
Semester I 2026/2027 Session 
(September Intake) 
21 September - 2 October 2026 International Student Orientation Programme  (ISOP) 
4 October 2026 - 9  

--- utm_academic_regulations.pdf | page 40 ---
23
Section 4
must obtain faculty approval for courses eligible for credit transfer in the 
semester before participating in the mobility program.
(9) Mobility students must inform the faculty of  the grades obtained for all 
courses taken as soon as they receive the course evaluation results from 
the higher education institutions.  
(10) For equivalent programmes, horizontal credit transfer 

In [8]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities
# Run ONCE after Cell 1 and Cell 3
from pathlib import Path
import json, hashlib
from datetime import datetime, timezone

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(f"Source not found: {selected}")
        return selected
    matches = sorted(p for p in RAW_DIR.rglob("*") if p.is_file()
                     and (filename is None or p.name == filename)
                     and (suffixes is None or p.suffix.lower() in suffixes))
    print("Candidates:", [str(p.relative_to(RAW_DIR)) for p in matches])
    if len(matches) != 1:
        raise ValueError("Select exactly one source. Set relative_path to an exact file under RAW_DIR.")
    return matches[0]

def source_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def save_manifest(records, extra=None):
    manifest = {"pipeline_version": "week3-v1.3",
                "generated_at_utc": datetime.now(timezone.utc).isoformat(),
                "records": records}
    manifest.update(extra or {})
    target = BASE_DIR / "data" / "week3_manifest_version1.3.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved lineage:", target)
    return manifest

print("Cell 4 completed successfully. Available: select_source(), source_hash(), save_manifest()")

Cell 4 completed successfully. Available: select_source(), source_hash(), save_manifest()


In [9]:
# CELL 4A — COMMON LINEAGE REGISTRY  (unchanged from v1.0)
from pathlib import Path
lineage_entries = {}
def register_lineage(source, output, processing):
    source = Path(source).resolve(); output = Path(output).resolve()
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    if not output.is_file():
        raise FileNotFoundError(f"Prepared output not found: {output}")
    if output.stat().st_size == 0:
        raise ValueError(f"Prepared output is empty: {output}")
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError("A processing description is required.")
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError("Replace the placeholder with the actual method.")
    source.relative_to(RAW_DIR.resolve()); output.relative_to(PROCESSED_DIR.resolve())
    lineage_entries[(str(source), str(output))] = {"source": source, "output": output, "processing": processing}
    print("Registered:", source.name, "->", output.name)
print("Common lineage registry ready.")

Common lineage registry ready.


In [10]:
# Cell 5A0 — Source configuration (v1.3: seven sources, S09 added)
from pathlib import Path
import re, json

MONTHS = "January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sept|Sep|Oct|Nov|Dec"

SOURCES = [
    dict(source_id="S01", filename="utm_academic_calendar_2026_2027_1.pdf", source_type="dates",
         language_default="en", language_by_page={},
         doc_date="2026-11-20", doc_date_basis="file name 'utm_academic_calendar_2026_2027_1' (session 2026/2027); no publication date printed",
         retrieved_date="2026-11-20", source_url="https://amd.utm.my/wp-content/uploads/sites/460/2026/05/Final_Kalendar-Akademik-Sarjana-Muda.Sarjana.Kedoktoran-20262027-14.5.2026-pembetulan.pdf",
         edition="Academic Calendar 2026/2027, Bachelor/Master/Doctoral (English, 1 page)",
         currency="current_2026", valid_until="2027-10-03", page_offset=0,
         drop=[r"^www\.utm\.my$", r"^DATE PROGRAMME PUBLIC HOLIDAY", r"^ACADEMIC CALENDAR FOR",
               r"^BACHELOR, MASTER AND DOCTORAL", r"^UNIVERSITI TEKNOLOGI MALAYSIA$", r"^Innovating Sustainable Solutions$"],
         fixups=[dict(name="rejoin_split_word_august", pattern=r"2 Au\s*\ngust 2027", repl="2 August 2027")],
         splitter="calendar", size=1200, chunk_notes={},
         note="Two-column table flattened by text extraction: read dates by neighbouring line. Undated PDF print."),
    dict(source_id="S02", filename="utm_academic_regulations.pdf", source_type="rules",
         language_default="en", language_by_page={},
         doc_date="2026", doc_date_basis="printed (copyright page 'First Edition 2026')",
         retrieved_date="2026-11-10", source_url="https://ugs.utm.my/wp-content/uploads/sites/8/2026/05/FT-UG-ACADEMIC-REGULATIONS-16th-edition.pdf",
         edition="Academic Regulations, Full-Time UG (title page: 16th Edition; copyright page: First Edition 2026)",
         currency="current_2026", valid_until=None, page_offset=17,
         drop=[], fixups=[], splitter="paragraph", size=800, chunk_notes={},
         note="PDF page = printed page + 17. Edition label conflicts between title page and copyright page."),
    dict(source_id="S05", filename="utm_User-Manual_Course-Registration-Senior-Student_update202620271.pdf", source_type="how_to",
         language_default="en", language_by_page={},
         doc_date="2026-10-06", doc_date_basis="file name 'update202620271' (session 2026/2027); PDF creation date",
         retrieved_date="2026-11-10", source_url="https://amd.utm.my/wp-content/uploads/sites/460/2026/10/User-Manual_Course-Registration-Senior-Student_update202620271.pdf",
         edition="Senior-student course registration manual (update 2026/2027)",
         currency="current_2026_student_verified", valid_until=None, page_offset=0,
         drop=[r"^www\.utm\.my$"], fixups=[], splitter="paragraph", size=1200, chunk_notes={},
         note="Slide deck. Currency verified by the student (evidence to be recorded). Screenshots (p3, p8) are not indexed; p3 shows a matric number."),
    dict(source_id="S06", filename="utm_ug_work_schedule_2026_2027.pdf", source_type="dates",
         language_default="en", language_by_page={},
         doc_date="2026-10-09", doc_date_basis="printed ('Date : 9 October 2026')",
         retrieved_date="2026-11-20", source_url="https://amd.utm.my/wp-content/uploads/sites/460/2026/10/Student-202620271-UG-Work-Schedule-for-Course-Registration.Award-of-Degree.Examination-1.pdf",
         edition="UG Work Schedule for Course Registration, Award of Degree, Examination (Semester I 2026/2027)",
         currency="current_2026_time_bound", valid_until="2027-03-21", page_offset=0,
         drop=[r"^NO\. ACTIVITY DURATION WEEK DATE REMARKS$"],
         fixups=[dict(name="drop_page_number_line", pattern=r"(?s)\A\d\s*\n(?=NO\. ACTIVITY)", repl=""),
                 dict(name="split_row3_from_row2", pattern=r"Deputy Dean\.(3) Course", repl=r"Deputy Dean.\n\1 Course")],
         splitter="rows", size=1200,
         chunk_notes={"Row 3": "Two date ranges listed; the PDF marks the revised range in blue (colour lost in text extraction).",
                      "Row 7": "Over 18 credits needs Academic Advisor and Deputy Dean approval; S02 Art. 13 says Dean approval.",
                      "Row 14": "Source typo: Thaipusam printed as '22 Janaury 2026' (calendar S01: 22 January 2027). 1 Jan 2027 labelled Federal Territory Day (KL only); S01 lists 1 Feb 2027 as Federal Territory Day and 1 Jan as New Year's Day (KL only).",
                      "Row 18": "Semester II registration window 11-22 January 2027."},
         note="Semester I 2026/2027 only. Time-bound: several deadlines have already passed after 24 Nov 2026. Source typos kept verbatim."),
    dict(source_id="S09", filename="utm_ug_work_schedule_sem2_2026_2027.pdf", source_type="dates",
         language_default="en", language_by_page={},
         doc_date="2026-09-10", doc_date_basis="pdf_creation_date (printed 'Date : September 2026', no day)",
         retrieved_date="2026-11-24", source_url="https://amd.utm.my/wp-content/uploads/sites/460/2026/09/Student-202620272-UG-Work-Schedule-for-Course-Registration.Award-of-Degree.Examination.pdf",
         edition="UG Work Schedule for Course Registration, Award of Degree, Examination (Semester II 2026/2027)",
         currency="current_2026_time_bound", valid_until="2027-10-03", page_offset=0,
         drop=[r"^NO\. ACTIVITY DURATION WEEK DATE REMARKS$"],
         fixups=[dict(name="drop_page_number_line", pattern=r"(?s)\A\d\s*\n(?=NO\. ACTIVITY)", repl="")],
         splitter="rows", size=1200,
         chunk_notes={"Row 1": "Text says registration 'in January 2026' for a Semester II 2026/2027 schedule; looks copied from an older edition (not corrected).",
                      "Row 7": "Over 18 credits needs Academic Advisor and Deputy Dean approval; S02 Art. 13 says Dean approval.",
                      "Row 17": "Titled 'Class Schedule for Semester I, 2026/2027'; in a Semester II schedule this probably means Semester I 2027/2028 (not corrected).",
                      "Row 9": "Source typo 'Submisssion' kept verbatim."},
         note="Semester II 2026/2027 only; companion to S06 (Semester I). Time-bound until the Semester II break ends. Source typos kept verbatim."),
    dict(source_id="S07", filename="utm_matric_card_faq_2026.pdf", source_type="faq",
         language_default="en", language_by_page={},
         doc_date="2026-10-01", doc_date_basis="printed ('Updated 1 Oktober 2026')",
         retrieved_date="2026-11-20", source_url="https://studentaffairs.utm.my/wp-content/uploads/2026/10/FAQs-Kad-Matrik-2026.pdf",
         edition="FAQ Matric Card (updated 1 Oct 2026)",
         currency="current_2026", valid_until=None, page_offset=0,
         drop=[r"^Updated \d+ \w+ \d{4}$"], fixups=[], splitter="qa", size=1200, chunk_notes={},
         note="Q&A pairs; mixes English with one Malay month name. Contains public office e-mail addresses only."),
    dict(source_id="S08", filename="utm_utmid_faq_2026.pdf", source_type="faq",
         language_default="en", language_by_page={},
         doc_date="2026-09-29", doc_date_basis="printed ('Updated on 29th September 2026')",
         retrieved_date="2026-11-20", source_url="https://studentaffairs.utm.my/wp-content/uploads/2026/09/UTMID-2026.pdf",
         edition="FAQ UTMID (updated 29 Sep 2026)",
         currency="current_2026", valid_until=None, page_offset=0,
         drop=[r"^UPDATED ON 29TH\s+SEPTEMBER 2026$"],
         fixups=[dict(name="rejoin_split_Q7_label", pattern=r"(?m)^(Q\d+)[ \t]*(?:\r?\n[ \t]*)*:[ \t]*(?:\r?\n[ \t]*)*", repl=r"\1 : ")],
         splitter="qa", size=1200, chunk_notes={},
         note="Q6 and Q7 answers are infographic images; supplemented by OCR + manual check (see OCR_SUPPLEMENTS)."),
]

# Retired from v1.0 (kept out of the corpus on purpose)
RETIRED = {
    "S03": "2010 advising handbook (AIMS 2000 era). Quarantined: outdated and conflicts with 2026 rules.",
    "S04": "New-student registration manual. Dropped: image-heavy and overlaps the senior-student manual (S05).",
}

for s in SOURCES:
    s["path"] = RAW_DIR / s["filename"]
    assert s["path"].is_file(), f"{s['source_id']}: file not found: {s['filename']}"
    s.setdefault("source_url", None)        # to be filled by the student (web-print PDFs carry no URL)
    s.setdefault("retrieved_date", None)    # to be filled by the student
    print(s["source_id"], "->", s["path"].name, "|", s["source_type"], "|", s["currency"])

S01 -> utm_academic_calendar_2026_2027_1.pdf | dates | current_2026
S02 -> utm_academic_regulations.pdf | rules | current_2026
S05 -> utm_User-Manual_Course-Registration-Senior-Student_update202620271.pdf | how_to | current_2026_student_verified
S06 -> utm_ug_work_schedule_2026_2027.pdf | dates | current_2026_time_bound
S09 -> utm_ug_work_schedule_sem2_2026_2027.pdf | dates | current_2026_time_bound
S07 -> utm_matric_card_faq_2026.pdf | faq | current_2026
S08 -> utm_utmid_faq_2026.pdf | faq | current_2026


In [11]:
# Cell 5A0b — Targeted OCR configuration (OCR only where pypdf cannot see answer content)
# Rule: one entry per image region that carries answer content. Nothing else is OCR'd.
# verified_text = text read from the page image by the student/assistant and checked; OCR raw is kept as evidence.
OCR_SUPPLEMENTS = [
    dict(supplement_id="OCR-S05-p009-legend", source_id="S05", pdf_page=9, attach="first_on_page",
         bbox=(0.28, 0.79, 0.72, 0.97), psm=6, method="ocr+manual_check",
         reason="Slip legend (learning-method and exam-type codes) exists only as an image.",
         verified_text=("Course registration slip legend. Learning Method: OL = Online Learning; FC = Face to face; "
                        "BL = Blended Learning (starts with online learning followed by face to face); BLS = Blended Learning Substitution. "
                        "Examination Type: AA = Alternative Assessment; NFE = No Final Exam; ASYNC O = Asynchronous online exam; "
                        "SYNC O = Synchronous online exam, no invigilation and no proctoring; SYNC OP = Synchronous online exam using online proctoring; "
                        "SYNC OV = Synchronous online exam with online invigilation; SYNC P = Synchronous online exam with physical invigilation; "
                        "PTF = Face-to-face final exam (printed as 'PTF' in the source; likely 'FTF', not verified).")),
    dict(supplement_id="OCR-S08-p001-Q6", source_id="S08", pdf_page=1, attach="section:Q6",
         bbox=(0.0, 0.0, 1.0, 1.0), psm=11, method="manual_transcription (ocr partial)",
         reason="Q6 answer (UTMSmart install steps) is an infographic; OCR recovers only fragments.",
         verified_text=("Answer (from infographic) - How to install UTMSmart. Step 1: Open Google Play Store or App Store. "
                        "Step 2: Download and install UTMSmart. Step 3: Open UTMSmart and click the menu 'Login'. "
                        "Step 4: Login using UTMID.")),
    dict(supplement_id="OCR-S08-p002-Q7", source_id="S08", pdf_page=2, attach="section:Q7",
         bbox=(0.0, 0.0, 1.0, 1.0), psm=11, method="manual_transcription (ocr partial)",
         reason="Q7 answer (Freshies' Guide steps) is an infographic.",
         verified_text=("Answer (from infographic) - How to access Freshies' Guide using UTMSmart. Step 1: Download and install UTMSmart. "
                        "Step 2: Click the 'Getting Around' menu. Step 3: Click the 'Freshies' Guide' menu. "
                        "Step 4: Click any Freshies' Guide menu.")),
]
# Never OCR these (contain personal data): S05 p3 (portal screenshot with a matric number), S05 p8 (screenshot).
OCR_FORBIDDEN = {("S05", 3), ("S05", 8)}
assert not any((o["source_id"], o["pdf_page"]) in OCR_FORBIDDEN for o in OCR_SUPPLEMENTS)

!pip -q install pytesseract
!apt-get -qq install -y tesseract-ocr poppler-utils > /dev/null
import subprocess, tempfile, difflib, shutil
try:
    import pytesseract
    from PIL import Image
    HAVE_OCR = shutil.which("tesseract") is not None and shutil.which("pdftoppm") is not None
except Exception:
    HAVE_OCR = False
print("OCR engine available:", HAVE_OCR)

def ocr_region(pdf_path, page, bbox, psm, dpi=300, scale=2):
    """Render one page, crop to bbox (fractions), upscale, run Tesseract. Returns raw text or None."""
    if not HAVE_OCR:
        return None
    with tempfile.TemporaryDirectory() as td:
        prefix = str(Path(td) / "pg")
        subprocess.run(["pdftoppm", "-r", str(dpi), "-f", str(page), "-l", str(page), "-png", "-singlefile",
                        str(pdf_path), prefix], check=True)
        im = Image.open(prefix + ".png"); w, h = im.size
        im = im.crop((int(bbox[0]*w), int(bbox[1]*h), int(bbox[2]*w), int(bbox[3]*h)))
        im = im.resize((im.width*scale, im.height*scale))
        txt = pytesseract.image_to_string(im, config=f"--psm {psm}")
    return " ".join(x.strip() for x in txt.splitlines() if x.strip())

PII_PATTERNS = {"matric": r"\b[AB]\d{2}[A-Z]{2}\d{4}\b", "ic12": r"\b\d{12}\b"}

OCR engine available: True


In [12]:
# Cell 5A1 — Extract, clean, chunk (two approaches compared) — v1.3
!pip -q install pypdf
from pypdf import PdfReader
import difflib

MIN_PAGE_CHARS = 20   # pages with less text than this are treated as image-only / blank

def clean_page(text, drop_patterns):
    text = re.sub(r"/gid\d+", "", text)
    keep = []
    for ln in (l.rstrip() for l in text.splitlines()):
        if any(re.match(p, ln.strip()) for p in drop_patterns):
            continue
        keep.append(ln)
    text = "\n".join(keep)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n[ ]+\n", "\n\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def split_units(text):
    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

def split_long(piece, size):
    """A sentence longer than size: break at line ends, then at spaces. Never truncate."""
    if len(piece) <= size: return [piece]
    out, buf = [], ""
    for ln in piece.split("\n"):
        for w in (ln.split(" ") if len(ln) > size else [ln]):
            sep = "\n" if (buf and w is ln) else " "
            if buf and len(buf) + len(sep) + len(w) <= size: buf += sep + w
            else:
                if buf: out.append(buf)
                buf = w
    if buf: out.append(buf)
    return out

SENT_END = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9(\u201c\"])")     # split only after a sentence end, never at a PDF line wrap

def pack(units, size):
    chunks, cur = [], ""
    for u in units:
        if len(u) > size:
            if cur:
                chunks.append(cur); cur = ""
            buf = ""
            for pc in (q for p in SENT_END.split(u) for q in split_long(p, size)):
                if len(buf) + len(pc) + 1 <= size:
                    buf += (" " if buf else "") + pc
                else:
                    if buf: chunks.append(buf)
                    buf = pc
            if buf: chunks.append(buf)
        elif len(cur) + len(u) + (2 if cur else 0) <= size:
            cur += ("\n\n" if cur else "") + u
        else:
            chunks.append(cur); cur = u
    if cur:
        chunks.append(cur)
    return chunks

# ---- source-specific splitters: each returns [(section_label, chunk_text), ...] ----
def split_calendar(text, size):
    """Calendar table: one chunk per block so every row stays with its heading."""
    pos = {}
    for key, pat in [("s1", r"(?m)^SEMESTER I \|"), ("s2", r"(?m)^SEMESTER II \|"), ("short", r"(?m)^SHORT SEMESTER \|")]:
        m = re.search(pat, text); pos[key] = m.start() if m else None
    marks = [m.start() for m in re.finditer(r"(?m)^International Undergraduate and", text)]
    if None in pos.values() or len(marks) < 2:
        return [(None, c) for c in pack(split_units(text), size)]        # fallback
    cuts = [(0, "Semester I intake: registration and orientation"),
            (pos["s1"], "Semester I 2026/2027 (12 Oct 2026 - 21 Mar 2027)"),
            (marks[-1], "Semester II intake: registration"),
            (pos["s2"], "Semester II 2026/2027 (22 Mar 2027 - 3 Oct 2027)"),
            (pos["short"], "Short Semester 2026/2027 (26 Jul 2027 - 3 Oct 2027)")]
    cuts.sort()
    out = []
    for i, (a, label) in enumerate(cuts):
        b = cuts[i + 1][0] if i + 1 < len(cuts) else len(text)
        seg = text[a:b].strip()
        if seg: out.append((label, seg))
    return out

_ROW = re.compile(rf"(?m)^(\d{{1,2}})[ \t]*(?:$|(?=[A-Z\"“]))(?!(?:{MONTHS})\b)")
def split_rows(text):
    """Work-schedule rows: split only at the expected next row number (avoids dates like '10 November')."""
    marks, exp = [], None
    for m in _ROW.finditer(text):
        n = int(m.group(1))
        if exp is None: exp = n
        if n == exp:
            marks.append((m.start(), n)); exp += 1
    if not marks:
        return [(None, text)]
    out = []
    for i, (a, n) in enumerate(marks):
        b = marks[i + 1][0] if i + 1 < len(marks) else len(text)
        seg = text[(0 if i == 0 else a):b].strip()
        out.append((f"Row {n}", seg))
    return out

def split_qa(text):
    parts = [p.strip() for p in re.split(r"(?m)^(?=Q\d+\b)", text) if p.strip()]
    out = []
    for p in parts:
        m = re.match(r"Q(\d+)\b", p)
        if m: out.append((f"Q{m.group(1)}", p))
        else:                                   # preamble (title line) is dropped, but must be short
            assert len(p) < 150, f"Unexpected long text before first Q: {p[:60]!r}"
    return out

def parse_printed_page(text):
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    for cand in ([lines[0], lines[-1]] if lines else []):
        if re.fullmatch(r"\d{1,3}", cand):
            return int(cand), cand
    return None, None

def section_of(text):
    for l in [l.strip() for l in text.splitlines() if l.strip()][:3]:
        m = re.search(r"Section\s+(\d+)", l)
        if m:
            return f"Section {m.group(1)}"
    return None

def extract_pages(src, s):
    reader = PdfReader(str(src)); pages, skipped, mism = [], [], []
    last_section = None
    for pno, page in enumerate(reader.pages, 1):
        raw = page.extract_text() or ""
        applied = []
        for fx in s["fixups"]:
            new = re.sub(fx["pattern"], fx["repl"], raw)
            if new != raw: applied.append(fx["name"]); raw = new
        printed, printed_line, inferred = None, None, False
        if s["page_offset"]:
            printed, printed_line = parse_printed_page(raw)
            if printed is not None and pno >= 18 and printed != pno - s["page_offset"]:
                mism.append((s["source_id"], pno, printed))
            if printed is None and pno >= 18:
                printed, inferred = pno - s["page_offset"], True
            sec = section_of(raw)
            if sec: last_section = sec
            if printed_line:
                raw = "\n".join(l for l in raw.splitlines() if l.strip() != printed_line)
            raw = "\n".join(l for l in raw.splitlines()
                            if not re.fullmatch(r"\s*(Academic Regulations\s+)?Section\s+\d+\s*", l))
        text = clean_page(raw, s["drop"])
        if len(text) < MIN_PAGE_CHARS:
            skipped.append(pno); continue
        pages.append(dict(pno=pno, text=text, printed=printed, section=last_section, inferred=inferred, fixups=applied))
    return pages, skipped, mism

def split_page(s, p):
    sp = s["splitter"]
    if sp == "calendar": return split_calendar(p["text"], s["size"])
    if sp == "rows":     return split_rows(p["text"])
    if sp == "qa":       return split_qa(p["text"])
    return [(p["section"], c) for c in pack(split_units(p["text"]), s["size"])]

def build_records(src, s, pages):
    sha = source_hash(src); recs = []
    for p in pages:
        for k, (label, ch) in enumerate(split_page(s, p), 1):
            note = s["note"]
            if label in s["chunk_notes"]:
                note += " | " + s["chunk_notes"][label]
            recs.append({
                "record_id": f"{s['source_id']}-p{p['pno']:03d}-{k:02d}",
                "source_id": s["source_id"],
                "source": str(Path(src).relative_to(RAW_DIR)),
                "source_sha256": sha,
                "source_type": s["source_type"],
                "source_url": s["source_url"],
                "retrieved_date": s["retrieved_date"],
                "edition": s["edition"],
                "doc_date": s["doc_date"],
                "doc_date_basis": s["doc_date_basis"],
                "valid_until": s["valid_until"],
                "currency_status": s["currency"],
                "language": s["language_by_page"].get(p["pno"], s["language_default"]),
                "pdf_page": p["pno"],
                "printed_page": p["printed"],
                "printed_page_inferred": p["inferred"],
                "section": label,
                "extraction_method": "pypdf",
                "text_fixups": p["fixups"],
                "reliability_note": note,
                "processing": f"page-aware; {s['splitter']} splitter; max_chars={s['size']}; footers removed; whitespace normalised",
                "text": ch,
            })
    return recs

def apply_supplements(records, supplements, sources):
    """Attach OCR/manual-checked text from image regions to an existing record (or fail loudly)."""
    log = []
    for o in supplements:
        s = next(x for x in sources if x["source_id"] == o["source_id"])
        pool = [r for r in records if r["source_id"] == o["source_id"] and r["pdf_page"] == o["pdf_page"]]
        if o["attach"].startswith("section:"):
            want = o["attach"].split(":", 1)[1]
            exact = [r for r in pool if r["section"] == want]
            if not exact and pool:
                print(f"WARNING {o['supplement_id']}: no record labelled {want} on page {o['pdf_page']}; "
                      f"found {[r['section'] for r in pool]}. Attaching to the first record on that page; check pypdf version.")
            pool = exact or pool
        assert pool, (f"No record to attach {o['supplement_id']}: page {o['pdf_page']} of {o['source_id']} produced no chunks. "
                      f"pypdf {__import__('pypdf').__version__}; page text starts: "
                      f"{(PdfReader(s['path']).pages[o['pdf_page']-1].extract_text() or '')[:120]!r}")
        tgt = pool[0]
        raw = ocr_region(s["path"], o["pdf_page"], o["bbox"], o["psm"])
        if raw:
            for k, pat in PII_PATTERNS.items():
                assert not re.search(pat, raw), f"PII pattern {k} in OCR output of {o['supplement_id']}"
        toks = lambda t: set(re.findall(r"\w+", t.lower()))
        # share of verified words that OCR also found (reading order is ignored: columns are often interleaved)
        sim = round(len(toks(raw) & toks(o["verified_text"])) / len(toks(o["verified_text"])), 2) if raw else None
        tgt["text"] = tgt["text"] + "\n\n[Text read from page image] " + o["verified_text"]
        tgt["extraction_method"] = "pypdf+" + o["method"]
        tgt["supplement_id"] = o["supplement_id"]
        tgt["ocr_raw_text"] = raw
        tgt["ocr_token_recall"] = sim
        log.append(dict(supplement_id=o["supplement_id"], record_id=tgt["record_id"], method=o["method"],
                        ocr_ran=raw is not None, ocr_chars=len(raw or ""), token_recall=sim))
    return log

prepared_records, evidence, mismatch_notes = [], [], []
for s in SOURCES:
    pages, skipped, mism = extract_pages(s["path"], s)
    mismatch_notes += mism
    s["skipped_pages"] = skipped
    recs = build_records(s["path"], s, pages)
    prepared_records += recs

    # ---- Approach A (baseline): fixed 500 characters over the whole document ----
    joined, boundaries = "", []
    for p in pages:
        boundaries.append(len(joined)); joined += p["text"] + "\n"
    A = [(i, joined[i:i + 500]) for i in range(0, len(joined), 500)]
    A_cross = sum(1 for i, c in A if any(i < b < i + len(c) for b in boundaries[1:]))
    A_mid = sum(1 for i, c in A if c[:1].islower())
    # how many Approach-B units (row / Q&A / calendar block) would Approach A cut in half?
    unit_split = None
    if s["splitter"] in ("calendar", "rows", "qa"):
        unit_split = 0
        for r in recs:
            at = joined.find(r["text"][:60])
            if at >= 0 and (at // 500) != ((at + len(r["text"]) - 1) // 500):
                unit_split += 1
    B_mid = sum(1 for r in recs if r["text"][:1].islower())
    evidence.append(dict(source=s["source_id"], pages=len(PdfReader(str(s["path"])).pages), skipped_pages=len(skipped),
                         A_chunks=len(A), A_cross_page=A_cross, A_start_lowercase=A_mid, A_units_cut=unit_split,
                         B_chunks=len(recs), B_cross_page=0, B_start_lowercase=B_mid))

ocr_log = apply_supplements(prepared_records, OCR_SUPPLEMENTS, SOURCES)

print("Printed-page mismatches vs offset 17 (inspect these):", mismatch_notes[:10])
print("\n--- Approach A (fixed 500 chars, whole document) vs B (page-aware, source-specific) ---")
for e in evidence: print(e)
print("\nOCR supplements:")
for o in ocr_log: print(" ", o)
print("\nTotal prepared records:", len(prepared_records))
print("Skipped image-only/blank pages:", {s["source_id"]: s["skipped_pages"] for s in SOURCES})

Printed-page mismatches vs offset 17 (inspect these): []

--- Approach A (fixed 500 chars, whole document) vs B (page-aware, source-specific) ---
{'source': 'S01', 'pages': 1, 'skipped_pages': 0, 'A_chunks': 7, 'A_cross_page': 0, 'A_start_lowercase': 3, 'A_units_cut': 4, 'B_chunks': 5, 'B_cross_page': 0, 'B_start_lowercase': 0}
{'source': 'S02', 'pages': 81, 'skipped_pages': 7, 'A_chunks': 205, 'A_cross_page': 71, 'A_start_lowercase': 156, 'A_units_cut': None, 'B_chunks': 183, 'B_cross_page': 0, 'B_start_lowercase': 18}
{'source': 'S05', 'pages': 10, 'skipped_pages': 0, 'A_chunks': 4, 'A_cross_page': 4, 'A_start_lowercase': 1, 'A_units_cut': None, 'B_chunks': 10, 'B_cross_page': 0, 'B_start_lowercase': 0}
{'source': 'S06', 'pages': 3, 'skipped_pages': 0, 'A_chunks': 10, 'A_cross_page': 2, 'A_start_lowercase': 7, 'A_units_cut': 9, 'B_chunks': 22, 'B_cross_page': 0, 'B_start_lowercase': 0}
{'source': 'S09', 'pages': 3, 'skipped_pages': 0, 'A_chunks': 10, 'A_cross_page': 2, 'A_start_lower

In [13]:
# Cell 5A2 — Export prepared_text_version1.3.jsonl and validate (B3 contract) — v1.3
import collections
approach = "page-aware-source-specific-splitting"
prepared_path = PROCESSED_DIR / "prepared_text_version1.3.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8", newline="\n") as out:
    for row in prepared_records:
        out.write(json.dumps(row, ensure_ascii=False) + "\n")

with prepared_path.open("r", encoding="utf-8") as fh:
    exported = [json.loads(l) for l in fh if l.strip()]

EXPECTED_UNITS = {"S01": 5, "S06": 22, "S09": 22, "S07": 11, "S08": 7}     # blocks / rows / Q&A pairs per source
assert exported and len(exported) == len(prepared_records), "Record count mismatch"
assert all(r["text"].strip() for r in exported), "Empty chunk found"
assert all(r["source_id"] and r["pdf_page"] for r in exported), "Missing provenance"
assert len({r["record_id"] for r in exported}) == len(exported), "Duplicate record IDs"
assert not any("/gid" in r["text"] for r in exported), "Glyph-code noise left in text"
assert not any(re.search(r"(?m)^www\.utm\.my$", r["text"]) for r in exported), "Footer left in text"
assert set(r["source_id"] for r in exported) == {s["source_id"] for s in SOURCES}, "A source is missing"
for sid, n in EXPECTED_UNITS.items():
    got = sum(r["source_id"] == sid for r in exported)
    assert got == n, f"{sid}: expected {n} units, got {got}"
assert not any(r["source_id"] in RETIRED for r in exported), "Retired source found in corpus"
print("Prepared file:", prepared_path, "| size:", prepared_path.stat().st_size, "bytes")
print("Records per source:", {sid: sum(r['source_id'] == sid for r in exported) for sid in sorted({r['source_id'] for r in exported})})
print("Currency labels:", sorted({(r['source_id'], r['currency_status']) for r in exported}))
print("Extraction methods:", dict(collections.Counter(r["extraction_method"] for r in exported)))
print("Fields still empty (to be filled by the student):",
      {k: sorted({r['source_id'] for r in exported if r[k] in (None, '')}) for k in ("source_url", "retrieved_date")})

# privacy scan (patterns only; read hits manually)
pats = {"email": r"[\w.-]+@[\w.-]+\.\w+", "phone": r"\b0\d-?\d{7,8}\b", "12-digit ID": r"\b\d{12}\b", "matric no.": r"\b[AB]\d{2}[A-Z]{2}\d{4}\b"}
for k, p in pats.items():
    hits = sorted({m for r in exported for m in re.findall(p, r["text"])})
    print("privacy:", k, "->", len(hits), hits[:8])
print("Validation: PASS (also read the sample records manually).")

Prepared file: /content/week3_project/data/processed/prepared_text_version1.3.jsonl | size: 393807 bytes
Records per source: {'S01': 5, 'S02': 183, 'S05': 10, 'S06': 22, 'S07': 11, 'S08': 7, 'S09': 22}
Currency labels: [('S01', 'current_2026'), ('S02', 'current_2026'), ('S05', 'current_2026_student_verified'), ('S06', 'current_2026_time_bound'), ('S07', 'current_2026'), ('S08', 'current_2026'), ('S09', 'current_2026_time_bound')]
Extraction methods: {'pypdf': 257, 'pypdf+ocr+manual_check': 1, 'pypdf+manual_transcription (ocr partial)': 2}
Fields still empty (to be filled by the student): {'source_url': [], 'retrieved_date': []}
privacy: email -> 6 ['digitalcare@utm.my', 'identitycard@utm.my', 'ugrad@utm.my', 'utmid@graduate.utm.my', 'utmid@utm.my', 'yourutmid@graduate.utm.my']
privacy: phone -> 0 []
privacy: 12-digit ID -> 0 []
privacy: matric no. -> 0 []
Validation: PASS (also read the sample records manually).


In [14]:
# B3 — basic output check (lab's required lines)
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:', prepared_path, 'Size:', prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

Output: /content/week3_project/data/processed/prepared_text_version1.3.jsonl Size: 393807
Validation: file exists and is non-empty; verify content manually too.


In [15]:
# B4.1 — Register lineage for each source (run in S01, S02, S05, S06, S09, S07, S08 order)
for s in SOURCES:
    register_lineage(source=s["path"], output=prepared_path,
                     processing=f"{approach}; splitter={s['splitter']}; max_chars={s['size']}; footers removed; whitespace normalised; "
                                f"OCR only for listed image regions")
print("Registered entries:", len(lineage_entries))

Registered: utm_academic_calendar_2026_2027_1.pdf -> prepared_text_version1.3.jsonl
Registered: utm_academic_regulations.pdf -> prepared_text_version1.3.jsonl
Registered: utm_User-Manual_Course-Registration-Senior-Student_update202620271.pdf -> prepared_text_version1.3.jsonl
Registered: utm_ug_work_schedule_2026_2027.pdf -> prepared_text_version1.3.jsonl
Registered: utm_ug_work_schedule_sem2_2026_2027.pdf -> prepared_text_version1.3.jsonl
Registered: utm_matric_card_faq_2026.pdf -> prepared_text_version1.3.jsonl
Registered: utm_utmid_faq_2026.pdf -> prepared_text_version1.3.jsonl
Registered entries: 7


In [16]:
# Cell 6A — Generate common lineage manifest (lab cell + source metadata) — v1.3
assert lineage_entries, "No lineage entries registered."
lineage_records = []
by_name = {s["path"].name: s for s in SOURCES}
for entry in lineage_entries.values():
    source, output, processing = entry["source"], entry["output"], entry["processing"]
    assert source.is_file() and output.is_file() and output.stat().st_size > 0
    cfg = by_name[source.name]
    lineage_records.append({
        "source_id": cfg["source_id"],
        "source_filename": str(source.relative_to(RAW_DIR.resolve())),
        "source_sha256": source_hash(source),
        "prepared_output": str(output.relative_to(BASE_DIR.resolve())),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_type": cfg["source_type"],
        "source_version": f"{cfg['edition']} ({cfg['doc_date']}; basis: {cfg['doc_date_basis']})",
        "currency_status": cfg["currency"],
        "valid_until": cfg["valid_until"],
        "source_url": cfg["source_url"],
        "retrieved_date": cfg["retrieved_date"],
        "access_rights": "Public official UTM document; coursework use only; raw file not redistributed",
        "known_limitations": cfg["note"],
        "skipped_pages": cfg["skipped_pages"],
        "ocr_supplements": [o["supplement_id"] for o in OCR_SUPPLEMENTS if o["source_id"] == cfg["source_id"]],
        "status": "Prepared; quality review pending",
    })
manifest = save_manifest(lineage_records, extra={"retired_sources": RETIRED, "ocr_log": ocr_log})
print("\nCOMMON LINEAGE MANIFEST GENERATED | entries:", len(manifest["records"]))

Saved lineage: /content/week3_project/data/week3_manifest_version1.3.json

COMMON LINEAGE MANIFEST GENERATED | entries: 7


In [17]:
# Cell 6B — Verify Common Lineage Manifest
import json
manifest_path = BASE_DIR / "data" / "week3_manifest_version1.3.json"
assert manifest_path.is_file(), "Common lineage manifest not found."
saved_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
saved_records = saved_manifest.get("records", [])
assert saved_records, "No lineage records found in saved manifest."
assert len(saved_records) == len(lineage_entries), "Saved lineage count does not match registration."
for record in saved_records:
    source = RAW_DIR / record["source_filename"]; output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], f"Source checksum mismatch: {source}"
    assert source_hash(output) == record["prepared_sha256"], f"Prepared output checksum mismatch: {output}"
print("COMMON LINEAGE VALIDATION: PASS | verified records:", len(saved_records))

COMMON LINEAGE VALIDATION: PASS | verified records: 7


In [18]:
# Cell 7 — REQUIRED: Common V1 pipeline utilities (unchanged logic; version label only)
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json

PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_RUNS = {}

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path
    if folder:
        if not source.is_dir(): raise FileNotFoundError(f"Source folder not found: {source}")
    elif not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    return source

def execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration):
    if kind == "images":
        source = Path(source)
        inputs = sorted(p for p in source.iterdir() if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"})
    elif kind == "documents":
        inputs = [Path(p) for p in source]
    else:
        inputs = [Path(source)]
    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError("No valid source files to ingest.")
    inspection = inspect_fn(inputs, configuration)                       # INSPECT
    output_dir = PIPE_DIR / kind; output_dir.mkdir(parents=True, exist_ok=True)
    outputs, mappings, method = prepare_fn(inputs, output_dir, configuration)   # PREPARE / STORE
    outputs = [Path(p) for p in outputs]
    if not outputs: raise ValueError("No prepared outputs were produced.")   # VALIDATE
    for output in outputs:
        if not output.is_file(): raise FileNotFoundError(f"Prepared output missing: {output}")
        if output.stat().st_size == 0: raise ValueError(f"Prepared output is empty: {output}")
    validation = {"output_count": len(outputs), "output_sha256": [sha256_file(p) for p in outputs],
                  "output_bytes": [p.stat().st_size for p in outputs]}
    record = {                                                           # RECORD LINEAGE
        "pipeline_version": "week3-v1.3", "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "data_type": kind, "configuration": configuration, "processing_method": method,
        "sources": [{"path": str(p), "sha256": sha256_file(p)} for p in inputs],
        "outputs": [str(p) for p in outputs], "source_output_mapping": mappings,
        "inspection": inspection, "validation": validation,
        "source_version": configuration.get("source_version", "Not verified"),
        "access_rights": configuration.get("access_rights", "Not verified - check before sharing"),
    }
    (output_dir / "pipeline_manifest_version1.3.json").write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"SUCCESS: {kind} | Inputs: {len(inputs)} | Outputs: {len(outputs)}")
    return record

def register_and_run(kind, source, inspect_fn, prepare_fn, configuration):
    def runner(): return execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration)
    first = runner()
    PIPELINE_RUNS[kind] = {"first": first, "runner": runner}
    return first
print("Cell 7 ready.")

Cell 7 ready.


In [ ]:
# Cell 8A — CONDITIONAL REQUIRED: PDF / TXT (UniGuide AI: seven official UTM PDFs) — v1.3
# Same helper functions as Cell 5A1 (Part B); only the pipeline stages are wrapped here.
public_cfg = lambda s: {k: v for k, v in s.items() if k != "path"}
CONFIG_A = {
    "method": approach, "min_page_chars": MIN_PAGE_CHARS,
    "source_version": "See docs/week3_data_inventory_version1.3.md (S01, S02, S05, S06, S09, S07, S08)",
    "access_rights": "Public official UTM documents; coursework use only; raw files not redistributed",
    "sources": [public_cfg(s) for s in SOURCES], "ocr_supplements": OCR_SUPPLEMENTS,
}
SOURCE_A = [resolve_source(s["filename"]) for s in CONFIG_A["sources"]]

def inspect_documents(inputs, cfg):
    report = {}
    for src in inputs:
        s = next(x for x in SOURCES if x["filename"] == Path(src).name)
        pages = [p.extract_text() or "" for p in PdfReader(str(src)).pages]
        chars = sum(len(t) for t in pages)
        if chars == 0:
            raise ValueError(f"{s['source_id']}: no extractable text (scanned PDF? needs OCR).")
        report[s["source_id"]] = {"file": Path(src).name, "pages": len(pages), "characters": chars,
            "low_text_pages": [i + 1 for i, t in enumerate(pages) if len(t.strip()) < cfg["min_page_chars"]],
            "gid_glyph_tokens": sum(len(re.findall(r"/gid\d+", t)) for t in pages)}
    return report

def prepare_documents(inputs, out, cfg):
    records, mapping = [], []
    target = out / "prepared_text_version1.3.jsonl"
    for src in inputs:
        s = next(x for x in SOURCES if x["filename"] == Path(src).name)
        pages, _, _ = extract_pages(src, s)
        recs = build_records(src, s, pages)
        if not recs: raise ValueError(f"{s['source_id']}: no chunks produced.")
        records += recs
        mapping.append({"source_id": s["source_id"], "source": str(src), "output": str(target), "records": len(recs)})
    apply_supplements(records, OCR_SUPPLEMENTS, SOURCES)
    with target.open("w", encoding="utf-8", newline="\n") as fh:
        for r in records: fh.write(json.dumps(r, ensure_ascii=False) + "\n")
    assert all(r["text"].strip() and r["source_id"] and r["pdf_page"] for r in records), "Provenance/text check failed"
    assert not any("/gid" in r["text"] for r in records), "Glyph codes left in text"
    return [target], mapping, f"{cfg['method']} (per-source splitter; see configuration)"

run_A = register_and_run("documents", SOURCE_A, inspect_documents, prepare_documents, CONFIG_A)

print("\nInspection:")
for sid, info in run_A["inspection"].items(): print(" ", sid, info)
print("\nSource -> output mapping:")
for m in run_A["source_output_mapping"]: print(" ", m["source_id"], "->", Path(m["output"]).name, "| records:", m["records"])

partB_path = BASE_DIR / "data" / "processed" / "prepared_text_version1.3.jsonl"
partC_path = Path(run_A["outputs"][0])
if partB_path.is_file():
    same = sha256_file(partB_path) == sha256_file(partC_path)
    print("\nPart C output identical to Part B output:", same)
    if not same: print("  Differences must be explained in docs/week3_pipeline_decisions_version1.3.md.")
sample = json.loads(next(l for l in partC_path.open(encoding="utf-8") if '"S02-p040' in l))
print("\nRepresentative record:", json.dumps({k: sample[k] for k in ("record_id", "pdf_page", "printed_page", "section", "currency_status", "extraction_method")}), "|", sample["text"][:150])
print("Cell 8A completed successfully.")

In [20]:
# Cell 9 — REQUIRED: Repeatability and provenance verification (unchanged)
assert PIPELINE_RUNS, "No pipeline runs registered. Run at least one selected Cell 8 pathway first."
print("--- Repeatability Verification ---")
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]; second = entry["runner"]()
    assert first["sources"] == second["sources"], f"Source fingerprint changed for {kind}"
    assert first["validation"] == second["validation"], f"Prepared output changed for {kind}"
    assert first["source_output_mapping"] == second["source_output_mapping"], f"Source-to-output mapping changed for {kind}"
    print(f"PASS: {kind} - source fingerprints, prepared outputs and mappings match")
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---
SUCCESS: documents | Inputs: 7 | Outputs: 1
PASS: documents - source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.
